In [1]:
import os
from multiprocess import set_start_method
set_start_method("spawn")
import logging
logger = logging.getLogger("ignite.handlers.early_stopping.EarlyStopping")
logger.setLevel(logging.WARNING)

In [2]:
from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from coreset_trainer.trainer import CoresetTrainer
import os
import torch

import argparse



In [3]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    optimizers
)
train_dataset, test_dataset, estimators = load_data_and_estimators()


In [6]:
%%writefile validation_engine.py
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
import numpy as np
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
from torch.cuda.amp import autocast
from tqdm import tqdm
import copy
from transformers import DataCollatorForLanguageModeling
from coreset_trainer.train import tokenize_dataset
from peft import LoraConfig, get_peft_model, PeftModel
from peft import PeftConfig
from finetune import CHAT_TEMPLATE
class ChatTemplateCollator:
    def __init__(self, tokenizer, mlm=False, max_length=1024): # TODO
        self.tokenizer = tokenizer
        
        self.max_length = max_length
        self.base_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer, mlm=mlm
        )

    def __call__(self, features):
        texts = [
            self.tokenizer.apply_chat_template(
                f["messages"], tokenize=False, add_generation_prompt=False
            )
            for f in features
        ]
        tokenized = self.tokenizer(
            texts,
            truncation=True,
            return_tensors="pt",
            padding=True,
            max_length=self.max_length
        )
 
        batch = [
            {"input_ids": tokenized["input_ids"][i],
             "attention_mask": tokenized["attention_mask"][i]}
            for i in range(len(tokenized["input_ids"]))
        ]
        return self.base_collator(batch)

class ValidationEngine():
    def __init__(self, adapter_path, device):

        self.device = device
        self.adapter_path = adapter_path
        self.tokenizer = AutoTokenizer.from_pretrained(self.adapter_path, padding_side='left')
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
    
        peft_config = PeftConfig.from_pretrained(adapter_path)
        self.base_model_path = peft_config.base_model_name_or_path

        
        self.data_collator = ChatTemplateCollator(tokenizer=self.tokenizer, mlm=False)
        if self.tokenizer.chat_template is None:
            self.tokenizer.chat_template = CHAT_TEMPLATE
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token # TODO
    def get_log_p(self, model, examples):
        batch = self.data_collator(examples)
        batch = {k: v.to(self.device) for k, v in batch.items()}
        with torch.no_grad(): 
            outputs = model.generate(**batch, max_new_tokens=5, return_dict_in_generate=True, output_scores=True,  num_beams=1)
            transition_scores = model.compute_transition_scores(
                outputs.sequences, outputs.scores, normalize_logits=True
            )
            log_p = transition_scores.sum(axis=1)
            # print("log_p",log_p)
            return log_p

            
        return self.model_
    def score(self, train_dataset, test_dataset):

        base_model_after_ft = AutoModelForCausalLM.from_pretrained(
                self.base_model_path,
                torch_dtype=torch.float32
            )
        model_after_ft = PeftModel.from_pretrained(base_model_after_ft, self.adapter_path).to(self.device)
        model_after_ft.config.pad_token_id = self.tokenizer.eos_token_id
        model_after_ft = self.finetune(model_after_ft, train_dataset)
        
        base_model_before_ft = AutoModelForCausalLM.from_pretrained(
                self.base_model_path,
                torch_dtype=torch.float32
            )
        model_before_ft = PeftModel.from_pretrained(base_model_before_ft, self.adapter_path).to(self.device)
        model_before_ft.config.pad_token_id = self.tokenizer.eos_token_id
        

        log_p_before_ft = self.get_log_p(model_before_ft, test_dataset)       
        log_p_after_ft = self.get_log_p(model_after_ft, test_dataset)
        return log_p_after_ft - log_p_before_ft

    def finetune(self, model, train_dataset):
        
        # Freeze the base model
        for param in model.base_model.parameters():
            param.requires_grad = False

        # Ensure LoRA adapters are trainable
        for name, param in model.named_parameters():
            if 'lora' in name:
                param.requires_grad = True
        
        training_args = TrainingArguments(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=1,
            remove_unused_columns=False,
            num_train_epochs=5,
            learning_rate=2e-5,
            logging_steps=500,
            seed=42,
            report_to=[],
            disable_tqdm=True
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            tokenizer=self.tokenizer,
            data_collator=self.data_collator,
        )

        trainer.train(resume_from_checkpoint=False)
        return model


Overwriting validation_engine.py


In [7]:
%%writefile validation_worker_validation.py
import os
import torch
from tqdm import tqdm
import pandas as pd

def process(partial_results_dir, estimator, explanation, examples_to_train_on, indices_to_train_on, examples_to_test_on, indices_to_test_on, train_dataset, train_dataset_name, train_dataset_split, test_dataset, test_dataset_name, test_dataset_split, device_id, ii):
        results_path = os.path.join(
            partial_results_dir,
            str(ii) + ".parquet",
            )
        os.makedirs(os.path.dirname(results_path), exist_ok=True)
        if os.path.isfile(results_path):
            print(f"Skipping {ii}: parquet file exists", flush=True)
            return
        else:
            try:
                # Restrict this process to a single GPU
                os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
                import torch
                from validation_engine import ValidationEngine  # import inside worker

                engine = ValidationEngine(
                                          estimator.model_path, 
                                          device="cuda")
                
                delta = engine.score(examples_to_train_on, examples_to_test_on)
                
                delta_target_document = delta[0].item()
                delta_random_documents = delta[1:].tolist()
                
                
                df = pd.DataFrame([(
                        explanation.description,
                        os.path.basename(estimator.model_path),
                        estimator.get_config_string(),
                        explanation.dataset_idx,
                        train_dataset_name,
                        train_dataset_split,
                        test_dataset_name,
                        test_dataset_split,
                        
                        indices_to_train_on,
                        
                        indices_to_test_on[0],
                        indices_to_test_on[1:],
                        
                        delta_random_documents,
                        delta_target_document
                    )], columns=[
                        "explanation_type", 
                        "model", 
                        "estimator",
                        "dataset_idx", 
                        "train_dataset", 
                        "train_split", 
                        "test_dataset", 
                        "test_split", 
                        
                        "indices_trained_on",
                        "indices_target_document",
                        "indices_random_documents",
                    
                        
                        "delta_random_documents",
                        "delta_target_document"
                        
                ])
                            
                assert df.notnull().all().all(), "DataFrame contains missing values"
                assert not df.isnull().values.any(), "DataFrame contains NaN values"
                df.to_parquet(results_path, index=False)
        
        
        

            except Exception as e:
                import traceback, logging
                logging.error(f"Error in process for GPU {device_id}: {e}")
                logging.error(traceback.format_exc())
                raise
        return []

Overwriting validation_worker_validation.py


In [8]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from tqdm import tqdm
import itertools
import pandas as pd
import traceback

logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')
multiprocessing.set_start_method('spawn', force=True)
torch.manual_seed(42)


n_test = 10

test_indices = test_dataset.shuffle(seed=42).select(range(n_test))["indices"]

num_devices = torch.cuda.device_count()







In [9]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from validation_worker_validation import process

logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')

multiprocessing.set_start_method('spawn', force=True)

num_gpus = torch.cuda.device_count()
logging.info(f"Detected {num_gpus} GPUs")

[INFO] Detected 2 GPUs


In [10]:

device_ids = itertools.cycle(range(num_devices))
results = []

from peft import LoraConfig, get_peft_model, PeftModel


for estimator in estimators:
    

    explanations = [
        explanation_type(row.name, estimator)
        for explanation_type in explanation_types
        for _, row in estimator.influence_estimate.iloc[test_indices].iterrows()
    ]
            
        
    partial_results_dir =  os.path.join("./cache/validation/partial/",
        estimator.get_config_string(),
        os.path.basename(estimator.model_path),
        train_dataset_name,
        train_dataset_split,
        test_dataset_name,
        test_dataset_split,
        "partial")
    with ProcessPoolExecutor(max_workers=4*num_devices) as executor:
        futures = {}
        for ii, explanation in enumerate(explanations):
            
            # random documents to test log_p for in addition to the document beeing explained
            random_test_indices = test_dataset.shuffle(seed=ii).select(range(5))["indices"]
        
            text_indices = [explanation.dataset_idx] + random_test_indices
            futures[explanation] = executor.submit(
                    process,
                    partial_results_dir,
                    estimator, explanation,
                    train_dataset.select(explanation.documents), explanation.documents, 
                    test_dataset.select(text_indices), text_indices,                 
                    train_dataset, train_dataset_name, train_dataset_split, test_dataset, test_dataset_name, test_dataset_split, 
                    next(device_ids),
                    ii
            )

        with tqdm(total=len(futures), desc="Explanations", position=0) as pbar:
            for future in as_completed(futures.values()):
                try:
                    # You can add future.result() here if you want to catch exceptions
                    future.result()  
                except Exception as e:
                    logging.error(f"A future failed: {e}\n{traceback.format_exc()}")
                    raise
                finally:
                    pbar.update(1)

Explanations:   0%|          | 0/50 [00:00<?, ?it/s]/srv/home/users/loriss21cs/cfe/validation_engine.py:124: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/srv/home/users/loriss21cs/cfe/validation_engine.py:124: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/srv/hom

{'train_runtime': 5.1382, 'train_samples_per_second': 9.731, 'train_steps_per_second': 9.731, 'train_loss': 4.510918273925781, 'epoch': 5.0}
{'train_runtime': 4.0023, 'train_samples_per_second': 12.493, 'train_steps_per_second': 12.493, 'train_loss': 4.534731750488281, 'epoch': 5.0}
{'train_runtime': 3.5449, 'train_samples_per_second': 14.105, 'train_steps_per_second': 14.105, 'train_loss': 4.534731750488281, 'epoch': 5.0}
{'train_runtime': 2.9207, 'train_samples_per_second': 17.119, 'train_steps_per_second': 17.119, 'train_loss': 4.7251370239257815, 'epoch': 5.0}
{'train_runtime': 2.9283, 'train_samples_per_second': 17.075, 'train_steps_per_second': 17.075, 'train_loss': 4.7251370239257815, 'epoch': 5.0}
{'train_runtime': 3.8327, 'train_samples_per_second': 13.046, 'train_steps_per_second': 13.046, 'train_loss': 4.534731750488281, 'epoch': 5.0}
{'train_runtime': 4.7649, 'train_samples_per_second': 10.493, 'train_steps_per_second': 10.493, 'train_loss': 4.510918273925781, 'epoch': 5.0}

Explanations:   0%|          | 0/50 [00:00<?, ?it/s]/srv/home/users/loriss21cs/cfe/validation_engine.py:124: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/srv/home/users/loriss21cs/cfe/validation_engine.py:124: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/srv/hom

{'train_runtime': 5.121, 'train_samples_per_second': 9.764, 'train_steps_per_second': 9.764, 'train_loss': 4.510918273925781, 'epoch': 5.0}
{'train_runtime': 3.5968, 'train_samples_per_second': 13.901, 'train_steps_per_second': 13.901, 'train_loss': 4.510918273925781, 'epoch': 5.0}
{'train_runtime': 3.9119, 'train_samples_per_second': 12.782, 'train_steps_per_second': 12.782, 'train_loss': 4.557433471679688, 'epoch': 5.0}
{'train_runtime': 3.0528, 'train_samples_per_second': 16.379, 'train_steps_per_second': 16.379, 'train_loss': 4.534731750488281, 'epoch': 5.0}
{'train_runtime': 2.555, 'train_samples_per_second': 19.57, 'train_steps_per_second': 19.57, 'train_loss': 4.7251370239257815, 'epoch': 5.0}
{'train_runtime': 3.1058, 'train_samples_per_second': 16.099, 'train_steps_per_second': 16.099, 'train_loss': 4.534731750488281, 'epoch': 5.0}
{'train_runtime': 5.3534, 'train_samples_per_second': 9.34, 'train_steps_per_second': 9.34, 'train_loss': 4.510918273925781, 'epoch': 5.0}
{'train_

Explanations:   0%|          | 0/50 [00:00<?, ?it/s][INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
[INFO] PyTorch version 2.7.0+cu126 available.
/srv/home/users/loriss21cs/cfe/validation_engine.py:124: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/srv/home/users/loriss21cs/cfe/validation_engine.py:124: FutureWarning: `tokenizer` is deprecated and will be 

{'train_runtime': 4.8238, 'train_samples_per_second': 10.365, 'train_steps_per_second': 10.365, 'train_loss': 4.510918273925781, 'epoch': 5.0}
{'train_runtime': 3.5286, 'train_samples_per_second': 14.17, 'train_steps_per_second': 14.17, 'train_loss': 4.510918273925781, 'epoch': 5.0}
{'train_runtime': 2.8019, 'train_samples_per_second': 17.845, 'train_steps_per_second': 17.845, 'train_loss': 4.488988952636719, 'epoch': 5.0}
{'train_runtime': 4.6719, 'train_samples_per_second': 10.702, 'train_steps_per_second': 10.702, 'train_loss': 4.364281311035156, 'epoch': 5.0}
{'train_runtime': 4.0857, 'train_samples_per_second': 12.238, 'train_steps_per_second': 12.238, 'train_loss': 4.54252197265625, 'epoch': 5.0}
{'train_runtime': 2.7905, 'train_samples_per_second': 17.918, 'train_steps_per_second': 17.918, 'train_loss': 5.007021484375, 'epoch': 5.0}
{'train_runtime': 5.0259, 'train_samples_per_second': 9.949, 'train_steps_per_second': 9.949, 'train_loss': 4.510918273925781, 'epoch': 5.0}
{'train

In [12]:
import pandas as pd

df_validation = pd.read_parquet("./cache/validation/partial/")

In [13]:
df_scoring = pd.read_parquet("./cache/scoring/partial/")

In [44]:
df = df_scoring.merge(
    df_validation,
    on=["explanation_type", "model", "estimator", "dataset_idx", "train_dataset", "train_split", "test_dataset", "test_split"],
    how="left"
)[["explanation_type", "model", "optimizer", "estimator", "dataset_idx", "pred_gain","mse", "delta_target_document"]]
df

,explanation_type,model,optimizer,estimator,dataset_idx,pred_gain,mse,delta_target_document
0,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,OptimizerKLT,DataInfEstimator: fast_implementation=True,59,1.229756,0.332690,-1.409365
1,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,OptimizerKLT,DataInfEstimator: fast_implementation=True,21,1.291397,0.106863,4.995133
2,Top-10 most influential,pythia-31m_tulu-v2-sft-mixture_train,OptimizerKLT,DataInfEstimator: fast_implementation=True,59,2.296454,0.178155,-2.008695
3,Top-10 most influential,pythia-31m_tulu-v2-sft-mixture_train,OptimizerKLT,DataInfEstimator: fast_implementation=True,21,1.305446,0.105713,4.060608
4,Top-10 most influential,pythia-31m_tulu-v2-sft-mixture_train,OptimizerKLT,DataInfEstimator: fast_implementation=True,56,1.550515,0.194724,-0.471478
...,...,...,...,...,...,...,...,...
295,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,OptimizerMSEL1,LESSEstimator: normalize=True,42,1.003162,0.000122,-0.426923
296,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,OptimizerMSEL1,LESSEstimator: normalize=True,50,0.000000,0.000000,0.385370
297,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,OptimizerMSEL1,LESSEstimator: normalize=True,27,1.002931,0.000122,-1.455358
298,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,OptimizerMSEL1,LESSEstimator: normalize=True,92,0.000000,0.000000,-0.475721


In [45]:
import statsmodels.api as sm

results = []


for keys, group in df.groupby(["explanation_type", "model", "estimator", "optimizer"]):
    x = pd.to_numeric(group["pred_gain"], errors="coerce")
    y = pd.to_numeric(group["delta_target_document"], errors="coerce")
    
    mask = ~(x.isna() | y.isna())
    x_clean, y_clean = x[mask], y[mask]
    

    X = sm.add_constant(x_clean)
    model = sm.OLS(y_clean, X).fit()
    
    results.append({
        "explanation_type": keys[0],
        "model": keys[1],
        "estimator": keys[2],
        "optimizer": keys[3],
        "coef": model.params.get("pred_gain", None),
        "p_value": model.pvalues.get("pred_gain", None),
        "r_squared": model.rsquared
    })


results_df = pd.DataFrame(results)
results_df


,explanation_type,model,estimator,optimizer,coef,p_value,r_squared
0,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,OptimizerKLT,-0.540479,0.853481,0.004527
1,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,OptimizerMSEL1,169.090723,0.193544,0.201180
2,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=False,OptimizerKLT,0.875100,0.381416,0.096853
3,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=False,OptimizerMSEL1,1.056342,0.390422,0.093425
4,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,OptimizerKLT,0.875298,0.381351,0.096878
5,10 random examples with seed 42,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=True,OptimizerMSEL1,1.060227,0.394506,0.091903
6,Bottom-10 least influential,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,OptimizerKLT,-3.780967,0.458693,0.070419
7,Bottom-10 least influential,pythia-31m_tulu-v2-sft-mixture_train,DataInfEstimator: fast_implementation=True,OptimizerMSEL1,112.452413,0.198762,0.197036
8,Bottom-10 least influential,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=False,OptimizerKLT,0.231457,0.861859,0.004020
9,Bottom-10 least influential,pythia-31m_tulu-v2-sft-mixture_train,LESSEstimator: normalize=False,OptimizerMSEL1,0.231457,0.861859,0.004020
